# BR 01 Entity Registry Screen

Author: Noah Green CPA CFE

Cross-Border Open-Source Diligence Atlas Lab

This notebook is a synthetic Brazil entity-registry source-literacy exercise. It uses a tiny fabricated CNPJ sample to show how intake identifiers can be normalized, checked for basic identifier quality, and converted into publication-safe leads for follow-up against official sources.

It does not query Receita Federal, does not establish that an entity exists or is active, does not make a legal conclusion, and does not score Brazil or any counterparty for jurisdiction-level risk.

## What this notebook does

- Loads `data/synthetic/brazil_entities.csv`.
- Uses `crossborder_dd.brazil_cnpj_parser.parse_cnpj_row` for CNPJ normalization and check-digit validation.
- Uses `crossborder_dd.country_registry_map.sources_by_country` to attach official-source literacy context.
- Writes a small redacted, synthetic lead table to `data/redacted_outputs/BR_01_entity_registry_screen.csv`.

All rows are synthetic and safe for public lab use.

In [1]:
from pathlib import Path
import csv
import sys


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src" / "crossborder_dd").exists() and (candidate / "data" / "synthetic").exists():
            return candidate
    raise RuntimeError("Could not find cross-border diligence atlas lab root")


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_PATH = str(PROJECT_ROOT / "src")
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

from crossborder_dd.brazil_cnpj_parser import parse_cnpj_row
from crossborder_dd.country_registry_map import sources_by_country
from crossborder_dd.export_csv import write_rows

DATA_PATH = PROJECT_ROOT / "data" / "synthetic" / "brazil_entities.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "redacted_outputs" / "BR_01_entity_registry_screen.csv"

DATA_PATH, OUTPUT_PATH

(PosixPath('/Users/noah.r.green/dd-tech-lab-companions/cross_border_diligence_atlas/data/synthetic/brazil_entities.csv'),
 PosixPath('/Users/noah.r.green/dd-tech-lab-companions/cross_border_diligence_atlas/data/redacted_outputs/BR_01_entity_registry_screen.csv'))

## Load the synthetic intake file

The input file is deliberately small. It contains fabricated entities only, including one identifier that should fail the CNPJ quality check.

In [2]:
with DATA_PATH.open(newline="", encoding="utf-8") as handle:
    synthetic_rows = list(csv.DictReader(handle))

synthetic_rows

[{'cnpj': '11222333000181', 'name': 'Empresa Sintetica Brasil Ltda'},
 {'cnpj': '00000000000000', 'name': 'Identificador Invalido SA'}]

## Parse CNPJ identifiers with the existing module

`identifier_valid` is an intake-quality signal only. A valid CNPJ check digit does not prove entity existence, legal status, authority, ownership, or current registry standing.

In [3]:
parsed_rows = [parse_cnpj_row(row) for row in synthetic_rows]
parsed_rows

[{'country': 'Brazil',
  'identifier': '11222333000181',
  'identifier_type': 'CNPJ',
  'identifier_valid': True,
  'entity_name': 'Empresa Sintetica Brasil Ltda',
  'entity_name_normalized': 'empresa sintetica brasil ltda'},
 {'country': 'Brazil',
  'identifier': '00000000000000',
  'identifier_type': 'CNPJ',
  'identifier_valid': False,
  'entity_name': 'Identificador Invalido SA',
  'entity_name_normalized': 'identificador invalido sa'}]

## Attach official-source literacy context

The country registry map tells a reviewer which official source family is relevant for the next step. This notebook does not scrape, query, or mirror the official source.

In [4]:
brazil_sources = sources_by_country("Brazil")
registry_source = next(source for source in brazil_sources if "CNPJ" in source.identifiers and "parsing" in source.lab_use)

{
    "source_name": registry_source.source_name,
    "access_method": registry_source.access_method,
    "identifiers": registry_source.identifiers,
    "lab_use": registry_source.lab_use,
    "limitation": registry_source.limitation,
}

{'source_name': 'Receita Federal Dados Abertos',
 'access_method': 'portal and bulk files',
 'identifiers': ('CNPJ',),
 'lab_use': 'CNPJ parsing on synthetic sample',
 'limitation': 'Bulk file metadata must be captured before any real-data use.'}

## Build the redacted lead table

The output table redacts entity names into synthetic row labels. The lead language stays limited to source-literacy next steps and data-quality issues.

In [5]:
redacted_rows = []
for index, parsed in enumerate(parsed_rows, start=1):
    identifier_quality = "valid_check_digit" if parsed["identifier_valid"] else "invalid_check_digit"
    lead = (
        "CNPJ format passes the check-digit test; verify against the official source before relying."
        if parsed["identifier_valid"]
        else "CNPJ format fails the check-digit test; correct intake data before registry follow-up."
    )
    redacted_rows.append(
        {
            "country": parsed["country"],
            "synthetic_subject_id": f"SYN-BR-{index:03d}",
            "entity_label": f"SYNTHETIC_BR_ENTITY_{index:03d}",
            "identifier_type": parsed["identifier_type"],
            "identifier_last4": parsed["identifier"][-4:],
            "identifier_quality": identifier_quality,
            "official_source_for_next_step": registry_source.source_name,
            "source_access_method": registry_source.access_method,
            "source_literacy_lead": lead,
            "required_escalation": "Escalate to Brazil local counsel before treating registry status, legal form, authority, or legal effect as established.",
            "publication_limitation": "Synthetic lead only; no registry query, entity-status finding, legal conclusion, jurisdiction-level score, or no-hit inference.",
        }
    )

redacted_rows

[{'country': 'Brazil',
  'synthetic_subject_id': 'SYN-BR-001',
  'entity_label': 'SYNTHETIC_BR_ENTITY_001',
  'identifier_type': 'CNPJ',
  'identifier_last4': '0181',
  'identifier_quality': 'valid_check_digit',
  'official_source_for_next_step': 'Receita Federal Dados Abertos',
  'source_access_method': 'portal and bulk files',
  'source_literacy_lead': 'CNPJ format passes the check-digit test; verify against the official source before relying.',
  'required_escalation': 'Escalate to Brazil local counsel before treating registry status, legal form, authority, or legal effect as established.',
  'publication_limitation': 'Synthetic lead only; no registry query, entity-status finding, legal conclusion, jurisdiction-level score, or no-hit inference.'},
 {'country': 'Brazil',
  'synthetic_subject_id': 'SYN-BR-002',
  'entity_label': 'SYNTHETIC_BR_ENTITY_002',
  'identifier_type': 'CNPJ',
  'identifier_last4': '0000',
  'identifier_quality': 'invalid_check_digit',
  'official_source_for_ne

In [6]:
write_rows(OUTPUT_PATH, redacted_rows)
OUTPUT_PATH

PosixPath('/Users/noah.r.green/dd-tech-lab-companions/cross_border_diligence_atlas/data/redacted_outputs/BR_01_entity_registry_screen.csv')

## What this proves / what it cannot prove

This proves that the lab can take a synthetic Brazil intake file, normalize CNPJ identifiers using shared code, catch an intentionally invalid identifier, connect the row to a documented official-source family, and produce a redacted lead table suitable for publication.

It cannot prove that any real entity exists, that a company is active or inactive, that a registry no-hit has meaning, that a person has authority to bind an entity, that ownership or control is known, or that any legal consequence follows from a row. Those questions require official-source review, source metadata, and Brazil local-counsel escalation before reliance.